In [1]:
import base64
import requests
from ultralytics import YOLO
from dao.BaseDao import BaseDao
import os
import cv2
import numpy as np
import shutil


In [13]:
def hsv_get(image_path):
    """
    再次分割书脊获得书标区域
    """
    # 读取图像
    image = cv2.imread(image_path)

    # 将图像转换为HSV颜色空间
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 定义红色的HSV范围
    lower_red = np.array([0, 100, 100])    # 红色的低阈值
    upper_red = np.array([10, 255, 255])   # 红色的高阈值

    # 创建一个mask，其中红色区域为白色，其他区域为黑色
    mask = cv2.inRange(hsv, lower_red, upper_red)

    # 寻找红线区域的轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 设置上下偏移量
    y_offset_top = -3
    y_offset_bottom = 17

    # 在原始图像上绘制红线区域的轮廓（仅作为示例）
    if contours:
        # 对轮廓按面积排序，取最大的两个轮廓
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

        # 获取两条红线的 y 坐标
        y_coords = []
        for contour in contours:
            _, y, _, _ = cv2.boundingRect(contour)
            y_coords.append(y)

        # 确定上下两条红线的 y 坐标并应用偏移量
        y_coords.sort()
        y_top = max(y_coords[0] - y_offset_top, 0)
        y_bottom = min(y_coords[1] + y_offset_bottom, image.shape[0])

        # 提取两条红线之间的区域
        between_region = image[y_top:y_bottom, :]

        # 保存结果
        shubiao = between_region
        # cv2.imwrite('output/hsv.jpg', between_region)

        print("两条最长红线之间的区域分割完成并保存")
    else:
        print("未找到红线区域，请调整阈值或检查图像")

In [12]:
hsv_get("shuji.jpg")

两条最长红线之间的区域分割完成并保存到 output/hsv.jpg


In [3]:
import os
results_with_question = dict()  # 存储存在疑问的结果
results_with_error = dict()  # 存储存在错误的结果
all_book_result_in_dict = dict()  # 存储所有书籍识别结果
# 清理旧的运行目录
if os.path.exists('./runs'):
    shutil.rmtree('./runs')
# 执行预处理步骤
dict_coordinate_data, sorted_coordinate_dicts, removed_id_dicts = preProcess()

# 请求并保存识别结果
print('确认请求 ...')

for name, sorted_dict in sorted_coordinate_dicts.items():
    all_book_result_in_list_dict = dict()  # 存储单个文件内所有书籍识别结果
    for key, xywh in sorted_dict:
        if key in removed_id_dicts[name]:  # 若ID已被移除，则跳过
            continue
        path = seq_to_filepath(name, key, result_dir)   # 构建文件路径  
        # 整合hsv_get函数
        hsv_get(path)
        hsv_path = 'out/between_regions_hsv.jpg'  # 由hsv_get函数保存的图像路径
        if os.path.exists(path):
            all_book_result_in_list_dict[key] = send_post_request(
                image_to_base64(path)
            ).json()  # 发送请求并获取响应JSON  存储识别结果
            # print('finish ' + str(key))
    all_book_result_in_dict[name] = all_book_result_in_list_dict  # 将单个文件的识别结果加入总结果字典

NameError: name 'preProcess' is not defined